# 02 Conversational Ai Assistant

#### Settings

In [6]:
from litellm import completion
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

import gradio as gr

messages = [
    {"role":"user", "content":"Hello!"}
]

response = completion(
    model="ollama/llama3.1:8b",
    messages=messages,
    api_base="http://localhost:11434"
)

print(response.choices[0].message.content)


Hello! It's nice to meet you. Is there something I can help you with or would you like to chat?


In [7]:
import os
import ollama
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

import gradio as gr

In [8]:
load_dotenv(override=True)
google_api_key = os.getenv("GOOGLE_API_KEY")
ollama_api_key = None

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/"

In [9]:
messages = [
    {"role":"user", "content":"Hello!"}
]

In [10]:
client_gemini = OpenAI(base_url=gemini_url, api_key=google_api_key)

response_gemini = client_gemini.chat.completions.create(model="gemini-2.5-flash-lite", messages=messages)

In [11]:
display(Markdown(response_gemini.choices[0].message.content))

Hello there! How can I help you today?

#### Gradio Interface

In [12]:
def cheers(text):
    # print(f"Cheeeersss!!! {text} !!!")
    return text.upper() + "~~~~!!!"

In [ ]:
message_input = gr.Textbox(label="Your message:", info="Enter a message to be cheered", lines=7)
message_output = gr.Textbox(label="Response:", lines=8)

view = gr.Interface(
    fn=cheers, 
    title="CHEERS!!!",
    inputs=[message_input], 
    outputs=[message_output], 
    examples=["Hello", "wuhoo"],
    flagging_mode="never"
    )

# view.launch(auth=("user", "pwd"), auth_message="Please Insert Your Username and Password.")
view.launch()

In [2]:
import requests
from litellm import completion
import gradio as gr

def get_ollama_models():
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            models = [f"ollama/{m['name']}" for m in response.json().get("models", [])]
            return models if models else ["ollama/gemma3:270m"]
    except Exception:
        return ["ollama/gemma3:270m"]

system_message = "You are a fun chat assistant! You will not only converse with the user, but also bring surprises in expressions and actions with emoji."

def message_local_llama(user_prompt, history, model_name):
    global messages
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": user_prompt}]

    response = completion(
        model=model_name,
        messages=messages,
        base_url="http://localhost:11434",
        temperature=1,
        stream=True
    )

    # print(response)
    partial_messages = ""
    for chunk in response:
        content = chunk.choices[0].delta.content
        if content:
            partial_messages += content
            yield partial_messages


available_models = get_ollama_models()

chat_room = gr.ChatInterface(
    fn=message_local_llama,
    additional_inputs=[
        gr.Dropdown(choices=available_models, value=available_models[0], label="Choose a model")
    ],
    additional_inputs_accordion=gr.Accordion(label="Model Settings", open=True),
    type="messages"
)

chat_room.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


### AI web summarizer

In [ ]:
# ai_web_summarizer.py
import gradio as gr
from litellm import completion
from scraper import fetch_website_contents

# ── Prompt ────────────────────────────────────────────────────────────────────

SYSTEM_MESSAGE = """
You are a news analyst assistant.
Given the raw text scraped from one or more news website homepages, produce a
structured summary of the headlines organised by category (e.g. Politics,
Economy, Technology, Sports, Health, Entertainment, World, etc.).

Rules:
- Write the output in Traditional Chinese (繁體中文).
- Format your response in Markdown (no code blocks).
- For each category, list the relevant headlines as a bullet list.
- If a headline has a clearly associated URL, append it as a Markdown link.
- Skip promotional / advertisement content.
- If multiple websites are provided, keep them separated by a level-2 heading
  with the site name / URL.
- Do not repeat a same headline in multiple category
"""

MODEL = "ollama/llama3.1:8b"          # fixed: was "llama3.1b:8b"
BASE_URL = "http://localhost:11434"


# ── Core streaming function ───────────────────────────────────────────────────

def stream_headlines(urls_text: str):
    """
    Accept a newline-separated list of URLs, scrape each one, and stream the
    LLM's categorised headline summary back to the Gradio Markdown component.
    """
    urls = [u.strip() for u in urls_text.strip().splitlines() if u.strip()]

    if not urls:
        yield "⚠️ 請輸入至少一個新聞網站 URL。"
        return

    # Build the user prompt
    prompt_parts = []
    for url in urls:
        content = fetch_website_contents(url)
        prompt_parts.append(f"### Source: {url}\n\n{content}")

    prompt = (
        "以下是從各新聞首頁擷取的原始文字，請依類別整理頭條新聞：\n\n"
        + "\n\n---\n\n".join(prompt_parts)
    )

    messages = [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user",   "content": prompt},   # fixed: was "User"
    ]

    response = completion(
        model=MODEL,
        messages=messages,
        base_url=BASE_URL,
        temperature=0.3,
        stream=True,
    )

    collected = ""
    for chunk in response:
        delta = chunk.choices[0].delta.content or ""
        collected += delta
        yield collected   # stream incrementally to Gradio


# ── Gradio UI ─────────────────────────────────────────────────────────────────

url_input = gr.Textbox(
    label="新聞網站 URL（每行一個）",
    placeholder="https://news.google.com/\nhttps://www.bbc.com/news\nhttps://www.cnn.com/",
    lines=5,
)

message_output = gr.Markdown(label="分類頭條摘要")

demo = gr.Interface(
    fn=stream_headlines,
    title="📰 新聞頭條爬取與分類",
    description=(
        "輸入一或多個新聞網站首頁 URL（每行一個），系統將自動擷取內容並由 AI 整理分類頭條新聞。"
    ),
    inputs=[url_input],
    outputs=[message_output],
    examples=[
        ["https://news.google.com/"],
        ["https://www.bbc.com/news\nhttps://www.cnn.com/"],
        ["https://news.google.com/\nhttps://www.reuters.com/\nhttps://www.bbc.com/news"],
    ],
    flagging_mode="never",
)

if __name__ == "__main__":
    demo.launch()

### Customer service chatbot

In [5]:
from litellm import completion

model_name = "ollama/llama3.1:8b"

# --- Business logic configuration (separated from code) ---

STORE_CONFIG = {
    "role": "You are a helpful assistant in a clothes store.",
    "goal": "Gently encourage the customer to try items that are on sale. If the customer is unsure what to get, recommend hats.",
    "discounts": {
        "Hats": "60% off",
        "Most other items": "50% off",
    },
    "unavailable_items": ["belts", "belt"],
}

SYSTEM_PROMPT_TEMPLATE = """
{role}

Your goal: {goal}

Current promotions:
{discounts}

Greeting rule:
If this is the very first message from the customer (i.e. there is no prior conversation history),
you MUST start your reply by warmly greeting the customer and briefly mentioning the current promotions.
Example: "Welcome! We're currently having a great sale — hats are 60% off and most other items are 50% off. How can I help you today?"

Example interaction:
- Customer: "I'm looking to buy a hat"
- You: "Wonderful - we have lots of hats - including several that are part of our sales event!"
""".strip()

MAX_HISTORY_TURNS = 10  # Keep recent rounds of conversation


# --- Prompt Construction ---

def build_system_prompt(user_message: str) -> str:
    discount_lines = "\n".join(
        f"- {item}: {rate}" for item, rate in STORE_CONFIG["discounts"].items()
    )

    base_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        role=STORE_CONFIG["role"],
        goal=STORE_CONFIG["goal"],
        discounts=discount_lines,
    )

    # Detect non-sale items (supports multiple keywords)
    message_lower = user_message.lower()
    mentioned_unavailable = [
        item for item in STORE_CONFIG["unavailable_items"]
        if item in message_lower
    ]

    if mentioned_unavailable:
        base_prompt += (
            "\n\nIMPORTANT: The store does NOT sell the following items the customer mentioned: "
            f"{', '.join(mentioned_unavailable)}. "
            "Politely let them know, and redirect them to similar or on-sale items we do carry."
        )

    return base_prompt


def trim_history(history: list, max_turns: int) -> list:
    """Retain the most recent max_turns rounds (each round = user + assistant) to avoid excessively long contexts."""
    max_messages = max_turns * 2
    return history[-max_messages:] if len(history) > max_messages else history


# --- generate greeting ---

def generate_greeting() -> str:
    """Let the AI ​​generate its own opening remarks, which will be called once when the interface starts."""
    discount_lines = "\n".join(
        f"- {item}: {rate}" for item, rate in STORE_CONFIG["discounts"].items()
    )
    base_prompt = SYSTEM_PROMPT_TEMPLATE.format(
        role=STORE_CONFIG["role"],
        goal=STORE_CONFIG["goal"],
        discounts=discount_lines,
    )

    response = completion(
        model=model_name,
        messages=[
            {"role": "system", "content": base_prompt},
            {"role": "user", "content": "[SYSTEM] The store just opened. Greet the customer warmly and mention current promotions."},
        ],
        base_url="http://localhost:11434",
        temperature=1,
    )
    return response.choices[0].message.content

# --- Main chat function ---

def chat(message: str, history: list):
    try:
        trimmed_history = trim_history(
            [{"role": h["role"], "content": h["content"]} for h in history],
            MAX_HISTORY_TURNS,
        )

        messages = (
            [{"role": "system", "content": build_system_prompt(message)}]
            + trimmed_history
            + [{"role": "user", "content": message}]
        )

        stream = completion(
            model=model_name,
            messages=messages,
            base_url="http://localhost:11434",
            temperature=1,
            stream=True,
        )

        response = ""
        for chunk in stream:
            response += chunk.choices[0].delta.content or ""
            yield response

    except Exception as e:
        yield f"⚠️We're sorry, there's a service error at the moment. Please try again later.（{type(e).__name__}: {e}）"


# --- start up ---

import gradio as gr

greeting = generate_greeting()

chatbot = gr.Chatbot(
    value=[{"role": "assistant", "content": greeting}],
    type="messages",
)

gr.ChatInterface(fn=chat, chatbot=chatbot, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
